In [ ]:
"""Personal Finance — Faker Code For Generation

Generation order:
  1 → Users, TransactionCategory
  2 → BankAccount, Budget, FinancialGoal, Loan, Investment,
            AIConversation, Notification
  3 → Transactions, Messages, InvestmentTracking,
            Stocks, Bonds, Deposit
  4 → UserNotification  (junction — last)"""

import csv
import os
import random
from datetime import date, timedelta, datetime
from faker import Faker

fake = Faker()
random.seed(42)
Faker.seed(42)

OUT = r"C:\Users\ASUS\OneDrive - National Center for Educational Technologies\Desktop\DB_data"
os.makedirs(OUT, exist_ok=True)

def write_csv(filename, rows, headers):
    path = os.path.join(OUT, filename)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=headers)
        w.writeheader()
        w.writerows(rows)
    print(f"  v  {filename:<48} {len(rows):>8,} rows")

def rand_date(years_ago=3):
    start = date.today() - timedelta(days=years_ago * 365)
    return start + timedelta(days=random.randint(0, years_ago * 365))

def future_date(min_days=30, max_days=1825):
    return date.today() + timedelta(days=random.randint(min_days, max_days))

def past_date(min_days=30, max_days=1095):
    return date.today() - timedelta(days=random.randint(min_days, max_days))

#SERIAL-style IDs
_counters = {}
def next_id(table):
    _counters[table] = _counters.get(table, 0) + 1
    return _counters[table]

# 1 — No FK dependencies

# ── Users ─────────────────────────────────────────────────────────────────────

NUM_USERS = 1_000
user_ids  = list(range(1, NUM_USERS + 1))
user_rows = []

for uid in user_ids:
    user_rows.append({
        "user_id":       uid,
        "first_name":    fake.first_name(),
        "surname":       fake.last_name(),
        "email":         fake.unique.email(),
        "phone":         fake.phone_number()[:50],
        "passwords":     fake.sha256()[:50],
        "nationality":    fake.country(),
        "last_login_at": str(fake.date_time_between("-30d", "now"))[:50],
        "status":        random.choices(["verified", "unverified"],
                                        weights=[85, 15])[0],
        "created_at":    fake.date_time_between("-3y", "now"),
    })

write_csv("01_users.csv", user_rows,
          ["user_id", "first_name", "surname", "email", "phone",
           "passwords", "natonality", "last_login_at", "status", "created_at"])

# ── TransactionCategory ───────────────────────────────────────────────────────

TX_CAT_DATA = [
    (1,  "Food & Dining",   "Restaurants, groceries, coffee"),
    (2,  "Transport",       "Fuel, taxis, public transit"),
    (3,  "Rent & Housing",  "Rent, mortgage, home maintenance"),
    (4,  "Salary",          "Monthly employment income"),
    (5,  "Utilities",       "Electricity, water, internet, gas"),
    (6,  "Health",          "Pharmacy, doctor visits, gym"),
    (7,  "Entertainment",   "Streaming, events, hobbies"),
    (8,  "Travel",          "Flights, hotels, vacation expenses"),
    (9,  "Shopping",        "Clothes, electronics, household items"),
    (10, "Education",       "Tuition, books, online courses"),
    (11, "Investment",      "Contributions to investment accounts"),
    (12, "Insurance",       "Health, car, home insurance premiums"),
    (13, "Gifts",           "Presents and donations"),
    (14, "Freelance",       "Income from freelance work"),
    (15, "Dividends",       "Dividend income from stocks"),
    (16, "Loan Payment",    "Monthly debt repayment"),
    (17, "Savings",         "Transfers into savings accounts"),
    (18, "Tax",             "Income tax, VAT payments"),
    (19, "Business",        "Business-related expenses"),
    (20, "Other",           "Uncategorised transactions"),
]

tx_cat_ids  = [r[0] for r in TX_CAT_DATA]
tx_cat_rows = [
    {"category_id": cid, "category_name": name, "description": desc}
    for cid, name, desc in TX_CAT_DATA
]

write_csv("02_transaction_category.csv", tx_cat_rows,
          ["category_id", "category_name", "description"])

# 2 — Depends only on Users (+ TransactionCategory for Budget)

BANKS      = ["Ameriabank", "Ardshinbank", "Inecobank", "ACBA Bank",
              "Evocabank", "VTB Armenia", "HSBC", "Citibank"]
ACCT_TYPES = ["checking", "savings", "credit", "investment"]
ACCT_NAMES = ["Main Account", "Savings Fund", "Travel Budget", "Emergency Reserve",
               "Investment Account", "Daily Expenses", "Joint Account"]
CURRENCIES = ["AMD", "USD", "EUR", "RUB", "GBP"]

# ── BankAccount ───────────────────────────────────────────────────────────────
account_ids  = []
account_rows = []

for uid in user_ids:
    for _ in range(random.randint(2, 4)):
        aid = next_id("account")
        account_ids.append(aid)
        account_rows.append({
            "account_id":   aid,
            "user_id":      uid,
            "account_name": random.choice(ACCT_NAMES),
            "bank_name":    random.choice(BANKS),
            "balance":      random.randint(0, 100_000),
            "currency":     random.choices(CURRENCIES, weights=[55, 20, 12, 8, 5])[0],
            "account_type": random.choice(ACCT_TYPES),
            "created_at":   fake.date_time_between("-3y", "now"),
        })

write_csv("03_bank_account.csv", account_rows,
          ["account_id", "user_id", "account_name", "bank_name",
           "balance", "currency", "account_type", "created_at"])

# ── Budget ────────────────────────────────────────────────────────────────────
BUDGET_NAMES = ["Groceries", "Rent", "Entertainment", "Travel", "Transport",
                "Health", "Utilities", "Dining Out", "Shopping", "Education"]

user_ids = list(range(1, 1001))
tx_cat_ids = list(range(1, 11))

def rand_date(years_ago=2):
    return fake.date_between(start_date=f"-{years_ago}y", end_date="today")

def next_id_generator():
    i = 1
    while True:
        yield i
        i += 1

budget_id_gen = next_id_generator()

def write_csv(filename, data, columns):
    path = os.path.join(OUT, filename)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(data)

budget_rows = []

for uid in user_ids:
    for _ in range(random.randint(2, 5)):
        start = rand_date()
        end = start + timedelta(days=random.randint(1, 365))
        amount_limit = random.randint(1, 5000)
        alert_threshold = random.randint(50, 92)
        threshold_value = int(amount_limit * alert_threshold / 100)
        consumed_amount = random.randint(0, threshold_value)
        budget_rows.append({
            "budget_id":       next(budget_id_gen),
            "user_id":         uid,
            "category_id":     random.choice(tx_cat_ids),
            "amount_limit":    amount_limit,
            "alert_threshold": alert_threshold,
            "start_date":      start,
            "end_date":        end,
            "status":          random.choices(["active", "not active"], weights=[70, 30])[0],
            "name":            random.choice(BUDGET_NAMES),
            "consumed_amount": consumed_amount
        })

write_csv(
    "04_budget.csv",
    budget_rows,
    [
        "budget_id", "user_id", "category_id", "amount_limit",
        "alert_threshold", "start_date", "end_date",
        "status", "name", "consumed_amount"
    ]
)

# ── FinancialGoal ─────────────────────────────────────────────────────────────

GOAL_NAMES = ["Emergency Fund", "Buy a Car", "Vacation", "Home Down Payment",
              "Pay Off Credit Card", "New Laptop", "Wedding Fund",
              "Retirement Savings", "Start a Business", "Study Abroad",
              "New Smartphone", "Move to New City"]
GOAL_TYPES = ["savings", "debt_payoff", "purchase", "investment", "emergency"]

goal_rows = []

for uid in user_ids:
    for _ in range(random.randint(2, 5)):
        status = random.choices(["active", "completed", "failed"],
                                 weights=[60, 25, 15])[0]
        target = random.randint(1, 30_000)

        if status == "completed":
            current = target
        elif status == "failed":
            current = random.randint(0, target // 2)
        else:
            current = random.randint(0, target)
        progress = int((current / target) * 100)
        deadline = (future_date(30, 1500) if status == "active"
                    else past_date(30, 730))

        goal_rows.append({
            "goal_id":        next_id("goal"),
            "user_id":        uid,
            "goal_name":      random.choice(GOAL_NAMES),
            "goal_type":      random.choice(GOAL_TYPES),
            "target_amount":  target,
            "current_amount": current,
            "progress_rate":  progress,
            "priority":       random.randint(1, 5),
            "deadline":       deadline,
            "status":         status,
            "created_at":     fake.date_time_between("-3y", "now"),
        })

write_csv("05_financial_goal.csv", goal_rows,
          ["goal_id", "user_id", "goal_name", "goal_type", "target_amount",
           "current_amount", "progress_rate", "priority", "deadline",
           "status", "created_at"])

# ── Loan ──────────────────────────────────────────────────────────────────────

LENDERS = ["Ameriabank", "Ardshinbank", "Inecobank", "ACBA Bank",
           "Private Lender", "VTB Armenia", "HSBC", "Evocabank"]

loan_ids  = []
loan_rows = []

for uid in user_ids:
    for _ in range(random.randint(1, 2)):
        lid       = next_id("loan")
        status    = random.choices(["active", "paid", "overdue"],
                                    weights=[60, 25, 15])[0]
        principal = random.randint(1_000, 50_000)
        start     = rand_date(years_ago=4)

        if status == "paid":
            remaining = 0
            due       = start + timedelta(days=random.randint(365, 1825))
        elif status == "overdue":
            remaining = int(principal * random.uniform(0.50, 0.99))
            due       = past_date(30, 730)
        else:
            remaining = int(principal * random.uniform(0.01, 0.99))
            due       = future_date(30, 1825)

        loan_ids.append(lid)
        loan_rows.append({
            "loan_id":          lid,
            "user_id":          uid,
            "lender_name":      random.choice(LENDERS),
            "principal_amount": principal,
            "remaining_amount": remaining,
            "interest_rate":    random.randint(4, 22),
            "start_date":       start,
            "due_date":         due,
            "status":           status,
        })

write_csv("06_loan.csv", loan_rows,
          ["loan_id", "user_id", "lender_name", "principal_amount",
           "remaining_amount", "interest_rate", "start_date", "due_date", "status"])

# ── Investment ────────────────────────────────────────────────────────────────

NUM_INVESTMENTS = 4_000
n_stock   = int(NUM_INVESTMENTS * 0.50)
n_bond    = int(NUM_INVESTMENTS * 0.30)
n_deposit = NUM_INVESTMENTS - n_stock - n_bond

subtypes = ["stock"] * n_stock + ["bond"] * n_bond + ["deposit"] * n_deposit
random.shuffle(subtypes)

INV_STATUSES = ["active", "sold", "matured", "cancelled"]
inv_meta      = []

investment_rows = []
for subtype in subtypes:
    iid      = next_id("investment")
    purchased = rand_date(years_ago=5)
    invested  = random.randint(1, 25_000)

    investment_rows.append({
        "investment_id":   iid,
        "user_id":         random.choice(user_ids),
        "investment_type": subtype,
        "amount_invested": invested,
        "expected_return": int(invested * random.uniform(1.01, 1.50)),
        "risk_level":      random.choice(["low", "medium", "high"]),
        "purchase_date":   purchased,
        "due_date":        future_date(180, 3650),
        "status":          random.choices(INV_STATUSES, weights=[65, 20, 10, 5])[0],
    })
    inv_meta.append({
        "investment_id": iid,
        "subtype":       subtype,
        "amount":        invested,
        "purchase_date": purchased,
    })

write_csv("07_investment.csv", investment_rows,
          ["investment_id", "user_id", "investment_type", "amount_invested",
           "expected_return", "risk_level", "purchase_date", "due_date", "status"])

# ── AIConversation ────────────────────────────────────────────────────────────

CONV_NAMES = ["Budget Review", "Investment Advice", "Savings Plan", "Loan Check",
              "Expense Analysis", "Goal Tracking", "Tax Planning", "Portfolio Check",
              "Spending Habits", "Monthly Summary"]
conv_ids  = []
conv_rows = []
for uid in user_ids:
    for _ in range(random.randint(1, 3)):
        cid = next_id("conv")
        conv_ids.append((cid, uid))
        conv_rows.append({
            "interaction_id":    cid,
            "user_id":           uid,
            "conversation_name": random.choice(CONV_NAMES) + " " + fake.month_name(),
            "priority":          random.choices(["pinned", "out pinned"],
                                                weights=[30, 70])[0],
            "conversation_date": fake.date_time_between("-1y", "now"),
        })

write_csv("08_ai_conversation.csv", conv_rows,
          ["interaction_id", "user_id", "conversation_name",
           "priority", "conversation_date"])

# ── Notification ──────────────────────────────────────────────────────────────

NOTIF_TITLES = [
    "Budget Alert", "Goal Reached", "Payment Due", "Loan Overdue",
    "New Transaction", "Weekly Summary", "Investment Update",
    "Savings Milestone", "Unusual Activity", "Deposit Maturity",
]
NOTIF_MSGS = [
    "Your budget has reached 80% of the limit.",
    "Congratulations! Your goal is now complete.",
    "Loan payment is due in 3 days. Ensure sufficient funds.",
    "Your loan is overdue. Please make a payment immediately.",
    "A new transaction was recorded on your account.",
    "Your weekly spending summary is ready for review.",
    "Your investment portfolio changed value this week.",
    "You reached a savings milestone this month!",
    "Unusual transaction detected. Please verify your account.",
    "Your fixed deposit matures in 30 days.",
]

notif_ids  = []
notif_rows = []

for uid in user_ids:
    for _ in range(random.randint(1, 3)):
        nid = next_id("notif")
        notif_ids.append((nid, uid))
        idx = random.randint(0, len(NOTIF_TITLES) - 1)
        notif_rows.append({
            "notification_id": nid,
            "user_id":         uid,
            "title":           NOTIF_TITLES[idx],
            "messages":        NOTIF_MSGS[idx],
            "is_read":         random.choices([True, False], weights=[60, 40])[0],
            "send_date":       fake.date_time_between("-6m", "now"),
        })

write_csv("09_notification.csv", notif_rows,
          ["notification_id", "user_id", "title", "messages", "is_read", "send_date"])

# 3 — Depends on Wave-2 entities
# ── Transactions ──────────────────────────────────────────────────────────────

TX_DESCS = [
    "Monthly grocery run", "Taxi to airport", "Online shopping",
    "Salary deposit", "Streaming subscription", "Doctor visit",
    "Restaurant dinner", "Hotel booking", "Gym membership", "Utility bill",
    "Freelance income", "ATM withdrawal", "Fuel top-up", "Insurance premium",
    "Book purchase", "Coffee shop", "Concert ticket", "Transfer to savings",
]

tx_ids  = []
tx_rows = []
NUM_TX  = 20_000

for _ in range(NUM_TX):
    tid    = next_id("tx")
    acc_id = random.choice(account_ids)
    other_accounts = [a for a in account_ids if a != acc_id]
    receiver_id = random.choice(other_accounts)
    loan_ref = random.choice(loan_ids) if random.random() < 0.10 else None

    tx_ids.append(tid)
    tx_rows.append({
        "transaction_id":   tid,
        "account_id":       acc_id,
        "loan_id":          loan_ref,
        "category_id":      random.choice(tx_cat_ids),
        "reciever_id":      receiver_id,
        "amount":           random.randint(1, 5_000),
        "status":           random.choices(["completed", "pending", "failed"],
                                           weights=[85, 10, 5])[0],
        "transaction_date": fake.date_time_between("-2y", "now"),
        "description":      random.choice(TX_DESCS),
        "exchange_rate":    random.choices([1, random.randint(300, 420)],
                                           weights=[70, 30])[0],
    })

write_csv("10_transactions.csv", tx_rows,
          ["transaction_id", "account_id", "loan_id", "category_id", "reciever_id",
           "amount", "status", "transaction_date", "description", "exchange_rate"])

# ── Messages ──────────────────────────────────────────────────────────────────

QUERIES = [
    "How much did I spend on food last month?",
    "Am I on track with my savings goal?",
    "What is my total loan balance?",
    "Show me my biggest expenses this week.",
    "How much interest will I pay on my current loan?",
    "Should I increase my budget for utilities?",
    "What is my net worth right now?",
    "Which account has the highest balance?",
    "How is my investment portfolio performing?",
    "When is my next loan payment due?",
]
RESPONSES = [
    "Based on your transactions, you spent AMD 85,000 on food last month.",
    "You are currently at 62% of your savings goal. Great progress!",
    "Your total outstanding loan balance is AMD 1,200,000.",
    "Your top three expenses this week were rent, groceries, and transport.",
    "At your current interest rate you will pay AMD 45,000 in interest total.",
    "Your utility spending is 15% above your set budget. Consider adjusting.",
    "Your estimated net worth based on accounts and investments is AMD 3,500,000.",
    "Your Ameriabank savings account has the highest balance at AMD 850,000.",
    "Your portfolio is up 4.3% this month, led by your tech stock positions.",
    "Your next loan payment of AMD 25,000 is due in 12 days.",
]

msg_rows = []

for cid, uid in conv_ids:
    for _ in range(random.randint(2, 8)):
        msg_rows.append({
            "message_id":         next_id("msg"),
            "interaction_id":     cid,
            "user_query":         random.choice(QUERIES),
            "assistant_response": random.choice(RESPONSES),
        })

write_csv("11_messages.csv", msg_rows,
          ["message_id", "interaction_id", "user_query", "assistant_response"])

# ── InvestmentTracking ────────────────────────────────────────────────────────

CHANGE_RANGES = {
    "stock":   (-8, 10),
    "bond":    (-2,  4),
    "deposit": ( 0,  1),
}

SNAPSHOTS_PER = 8

tracking_rows = []

for meta in inv_meta:
    iid     = meta["investment_id"]
    subtype = meta["subtype"]
    lo, hi  = CHANGE_RANGES[subtype]
    val     = meta["amount"]
    snap_dt = meta["purchase_date"]

    for _ in range(SNAPSHOTS_PER):
        snap_dt = snap_dt + timedelta(days=30)
        chg     = random.randint(lo, hi)
        val     = max(1, int(val * (1 + chg / 100)))
        snap_ts = datetime.combine(snap_dt, datetime.min.time())

        tracking_rows.append({
            "tracking_id":   next_id("tracking"),
            "investment_id": iid,
            "tracked_value": val,
            "change_percent": chg,
            "dates":         snap_ts,
        })

write_csv("12_investment_tracking.csv", tracking_rows,
          ["tracking_id", "investment_id", "tracked_value", "change_percent", "dates"])

# ── Stocks ────────────────────────────────────────────────────────────────────
STOCK_LIST = [
    ("Apple",     170, 220), ("Tesla",     140, 280),
    ("Amazon",    120, 200), ("Nvidia",    350, 950),
    ("Microsoft", 280, 450), ("Google",    120, 185),
    ("Meta",      280, 540), ("HSBC",       30,  70),
    ("Samsung",    45,  85), ("Alibaba",    60, 130),
    ("Berkshire", 290, 560), ("JPMorgan",  130, 210),
]

stock_inv_ids = [m["investment_id"] for m in inv_meta if m["subtype"] == "stock"]
stock_rows    = []

for iid in stock_inv_ids:
    name, lo, hi = random.choice(STOCK_LIST)
    stock_rows.append({
        "investment_id":    iid,
        "stock_name":       name,
        "current_price":    random.randint(lo, hi),
        "dividend_yield":   random.randint(0, 8),
        "number_of_shares": random.randint(1, 500),
    })

write_csv("13_stocks.csv", stock_rows,
          ["investment_id", "stock_name", "current_price",
           "dividend_yield", "number_of_shares"])

# ── Bonds ─────────────────────────────────────────────────────────────────────

COUPON_FREQS = ["annual", "semi_annual", "quarterly", "monthly"]
bond_inv_ids = [m["investment_id"] for m in inv_meta if m["subtype"] == "bond"]
purchase_lookup = {m["investment_id"]: m["purchase_date"] for m in inv_meta}
bond_rows = []

for iid in bond_inv_ids:
    purchase_dt = purchase_lookup[iid]
    maturity    = purchase_dt + timedelta(days=random.randint(365, 365 * 20))
    bond_rows.append({
        "investment_id":    iid,
        "maturity_date":    maturity,
        "coupon_frequency": random.choice(COUPON_FREQS),
        "bond_type":        random.choice(["corporate", "government"]),
    })

write_csv("14_bonds.csv", bond_rows,
          ["investment_id", "maturity_date", "coupon_frequency", "bond_type"])

# ── Deposit ───────────────────────────────────────────────────────────────────

deposit_inv_ids = [m["investment_id"] for m in inv_meta if m["subtype"] == "deposit"]
deposit_rows    = []

for iid in deposit_inv_ids:
    purchase_dt = purchase_lookup[iid]
    dep_type    = random.choice(["fixed_term", "savings", "cd", "money_market"])
    maturity    = (purchase_dt + timedelta(days=random.randint(90, 365 * 5))
                   if dep_type != "savings" else None)
    deposit_rows.append({
        "investment_id": iid,
        "bank_name":     random.choice(BANKS),
        "interest_rate": random.randint(1, 8),
        "maturity_date": maturity,
    })

write_csv("15_deposit.csv", deposit_rows,
          ["investment_id", "bank_name", "interest_rate", "maturity_date"])

# 4 — Junction tables (depend on everything above)

seen_un = set()
un_rows = []

for nid, uid in notif_ids:
    extra = random.sample(
        [u for u in user_ids if u != uid],
        k=random.randint(0, 2)
    )
    for extra_uid in extra:
        if (extra_uid, nid) not in seen_un:
            seen_un.add((extra_uid, nid))
            un_rows.append({"user_id": extra_uid, "notification_id": nid})

write_csv("16_user_notification.csv", un_rows,
          ["user_id", "notification_id"])



── Wave 1: Users, TransactionCategory ──
  v  01_users.csv                                        1,000 rows
  v  02_transaction_category.csv                            20 rows

── Wave 2: BankAccount, Budget, FinancialGoal, Loan, Investment, AIConversation, Notification ──
  v  03_bank_account.csv                                 2,910 rows
  v  04_budget.csv                                       3,545 rows
  v  05_financial_goal.csv                               3,457 rows
  v  06_loan.csv                                         1,498 rows
  v  07_investment.csv                                   4,000 rows
  v  08_ai_conversation.csv                              2,021 rows
  v  09_notification.csv                                 2,042 rows

── Wave 3: Transactions, Messages, InvestmentTracking, Stocks, Bonds, Deposit ──
  v  10_transactions.csv                                20,000 rows
  v  11_messages.csv                                    10,213 rows
  v  12_investment_tracking.cs